In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from time import time
import pandas as pd
import numpy as np
import tqdm
import random
from collections import defaultdict
import argparse

import torch
import torch.nn as nn
from torch import cuda
from torch.optim import Adam
from torch.utils.data import Dataset

from torchkge.sampling import BernoulliNegativeSampler
from torchkge.utils import MarginLoss,DataLoader
from torchkge import KnowledgeGraph,DistMultModel,TransEModel,TransRModel
from torchkge.models.bilinear import HolEModel,ComplExModel

from utils import *

/home/sandro/miniconda3/envs/kge5080/lib/python3.10/site-packages/torchkge/utils/data_redundancy.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
'''
This section is user defined !!!
'''
h_dim = 300

data_path = "../processed_data/target_inference_1/"
save_model_path = '../best_model/target_inference_1/'
output_path = "../results/target_inference_1/"

ent_list = ['CID:60184959', 'CID:9549284', 'CID:54660959', 'CID:49843139', 'CID:44142162', 'CID:60185257', 'CID:20862025', 'CID:3336', 'CID:4484064', 'CID:54657734', 'CID:2208391', 'CID:44617325', 'CID:16220015', 'CID:9902311', 'CID:24747177', 'CID:73707475', 'CID:9954280', 'CID:60185118', 'CID:60191079', 'CID:60190043']

In [ ]:


name = "target_inference_1"
data_path = f"../processed_data/{name}/"

ent2id = np.load(data_path + "ent2id.npy", allow_pickle=True).item()

compounds = sorted([k for k in ent2id.keys() if k.startswith("CID:")])
print("num compounds:", len(compounds))
print("first 20:", compounds[:20])
print("random 20:", random.sample(compounds, 20))

num compounds: 10892
first 20: ['CID:10000456', 'CID:100018', 'CID:10022508', 'CID:10040286', 'CID:100493', 'CID:10068193', 'CID:10071166', 'CID:10074640', 'CID:10077147', 'CID:1007755', 'CID:10108190', 'CID:10109069', 'CID:10109823', 'CID:10113978', 'CID:10125107', 'CID:10126189', 'CID:10127622', 'CID:10131112', 'CID:10133', 'CID:1013376']
random 20: ['CID:2793096', 'CID:54652699', 'CID:4946', 'CID:2998821', 'CID:44623884', 'CID:60184914', 'CID:54651370', 'CID:6603827', 'CID:54666528', 'CID:60185143', 'CID:44483932', 'CID:60191043', 'CID:54666592', 'CID:60189828', 'CID:5924208', 'CID:60185081', 'CID:24747192', 'CID:60185084', 'CID:54661015', 'CID:4792']


In [3]:
# load processed_data after training
cause = pd.read_csv(data_path + 'cause.txt',sep='\t',names=['from','rel','to'])
ent2id = np.load(data_path + 'ent2id.npy', allow_pickle=True).item()
rel2id = np.load(data_path + 'rel2id.npy', allow_pickle=True).item()

h_cand = [v for k,v in ent2id.items() if k.startswith('CID:')]
h_cand_ent = [k for k,v in ent2id.items() if k.startswith('CID:')]

t_cand = [v for k,v in ent2id.items() if k.startswith(('Protein:','TF:','RBP:'))]
t_cand_ent = [k for k,v in ent2id.items() if k.startswith(('Protein:','TF:','RBP:'))]

In [4]:
# count target inference score
ti_dict = {}

for ent in tqdm.tqdm(ent_list):
    results = []
    for i in range(5):
        model = DistMultModel(h_dim, len(ent2id), len(rel2id))
        model.load_state_dict(torch.load(save_model_path + "pertkg{}.pt".format(i)))
        # ckpt = torch.load(save_model_path + f"pertkg{i}.pt", map_location="cpu")
        # model.load_state_dict(ckpt)

# 后面再决定要不要上 GPU
        if torch.cuda.is_available():
            model = model.cuda()
        if cuda.is_available():
            cuda.empty_cache()
            model.cuda()
        model.normalize_parameters()
        model.eval()
        with torch.no_grad():
            ent_emb,rel_emb = model.get_embeddings() # (n_ent, emb_dim)
            score = inference(ent,
                            ent2id,rel2id,
                            ent_emb,rel_emb,
                            h_cand,t_cand,
                            'target_inference')
            results.append(score)

    average_list = [sum(x) / len(x) for x in zip(*results)]
    ti_dict['{}'.format(ent)] = average_list

100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


In [ ]:
# count confidence
# results = []
# import gc
# for i in range(5):
#     model = DistMultModel(h_dim, len(ent2id), len(rel2id))
#     model.load_state_dict(torch.load(save_model_path + "pertkg{}.pt".format(i)))
#     # ckpt = torch.load(save_model_path + f"pertkg{i}.pt", map_location="cpu")
#     # model.load_state_dict(ckpt)
#     if cuda.is_available():
#         cuda.empty_cache()
#         model.cuda()
#     model.normalize_parameters()
#     model.eval()
#     with torch.no_grad():        
#         ent_emb,rel_emb = model.get_embeddings() # (n_ent, emb_dim)
#         score = inference(h_cand_ent,
#                         ent2id,rel2id,
#                         ent_emb,rel_emb,
#                         h_cand,t_cand,
#                         'batch_target_inference')
#         if isinstance(score, torch.Tensor) and score.is_cuda:
#             score = score.cpu().numpy()
#             results.append(score)
#         results.append(score)
#     # 释放内存
#     del model, ent_emb, rel_emb
#     gc.collect()
#     torch.cuda.empty_cache()

# # model = DistMultModel(...)
# # for i in range(5):
# #     ckpt = torch.load(..., map_location='cpu')
# #     model.load_state_dict(ckpt)
# #     model.to(device)
# #     with torch.no_grad():
# #          ent_emb, rel_emb = model.get_embeddings()
# #          # move to cpu etc...
# #     model.to('cpu')   # 释放显存
# #     del ckpt
# #     gc.collect()
# #     torch.cuda.empty_cache()


# arr1 = np.array(results[0])
# arr2 = np.array(results[1])
# arr3 = np.array(results[2])
# arr4 = np.array(results[3])
# arr5 = np.array(results[4])
# average_arr = np.mean([arr1, arr2, arr3, arr4, 
#                        arr5,
#                        ], axis=0).tolist()

# ti_dict_n_n = {}
# for idx, ent in enumerate(h_cand_ent):
#     ti_dict_n_n['{}'.format(ent)] = average_arr[idx]

# ti_percent_dict = {}
# for k,v in tqdm.tqdm(ti_dict.items()):  
#     ti_percent = []

#     k_ranks = get_rank(v)

#     comp_ranks = []
#     for comp in h_cand_ent:  







#         comp_ranks.append(get_rank(ti_dict_n_n[comp]))
        
#     packed_ranks = list(zip(*comp_ranks))

#     for idx,x in enumerate(k_ranks):
#         ranks = packed_ranks[idx]
#         ti_percent.append(sum([1 for i in ranks if i >= (x+50)])/len(ranks))  # 50 is correct factor

#     ti_percent_dict[k] = ti_percent
#########
#优化：原代码不会自动清理模型的缓存，易导致机器内存溢出卡死，现在设计内存回收机制
import os, gc, tqdm
import numpy as np
import torch
from torch import cuda

# 如果你在 notebook 里已经设置 CUDA_VISIBLE_DEVICES，那这里用默认即可
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- 配置 / 预加载 ----
# 假设 h_dim, ent2id, rel2id, h_cand_ent, h_cand, t_cand 等变量已提前定义（和你原来一样）
# 我们只创建一个 model 实例并复用它
model = DistMultModel(h_dim, len(ent2id), len(rel2id))
# 把 model 放到 GPU（只做一次）
model.to(device)
model.eval()

# 可选：开启 cudnn.benchmark 有时候能加速（视情况）
torch.backends.cudnn.benchmark = True

# ---- 辅助函数 ----
def to_cpu_numpy(x):
    """把 tensor 或其他类型安全地转成 numpy array（如果已经是 numpy 则返回原样）。"""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    else:
        return np.array(x)

# ---- target inference（示例） ----
ti_dict = {}
for ent in tqdm.tqdm(ent_list, desc="target_inference (ent_list)"):
    results = []
    for i in range(5):
        # 1) 先把 checkpoint load 到 CPU，这样不会在加载时瞬间把大张量塞到 GPU（更稳健）
        ckpt = torch.load(save_model_path + f"pertkg{i}.pt", map_location="cpu")

        # 2) load_state_dict（model 已经在 GPU 上），PyTorch 会把 CPU 的参数复制到 model 的 GPU 参数中
        model.load_state_dict(ckpt)
        # 如果模型内部有需要 normalize 的操作
        model.normalize_parameters()
        model.eval()

        # 3) 在 GPU 上进行推理。确保 torch.no_grad()
        with torch.no_grad():
            # 如果你想用混合精度以减小显存和提速，可以用下面注释代码
            # with torch.cuda.amp.autocast():
            ent_emb, rel_emb = model.get_embeddings()   # 通常在 GPU 上（因为 model 在 GPU）
            # 直接把 GPU Tensor 传入你的 inference（如果 inference 支持 GPU 张量）
            score = inference(ent,
                              ent2id, rel2id,
                              ent_emb, rel_emb,
                              h_cand, t_cand,
                              'target_inference')

            # 关键：**马上把结果转为 CPU numpy**，不要把 GPU Tensor 放到 results 列表
            score_np = to_cpu_numpy(score)
            results.append(score_np)

        # 4) 清理本轮临时变量（确保没有对 GPU 张量的引用）
        del ckpt
        # ent_emb/rel_emb 可能是对 model 参数的引用或临时 tensor，删除引用以避免额外占用
        del ent_emb, rel_emb, score, score_np
        # 强制回收并清理 CUDA 缓存（注意 empty_cache 只释放缓存，不会删除被引用的张量）
        gc.collect()
        torch.cuda.empty_cache()

    # 5) 汇总结果（results 是 numpy 数组列表）
    average_list = np.mean(results, axis=0).tolist()
    ti_dict[f'{ent}'] = average_list

# ---- count confidence（你的原始单元） ----
# 由于我们已经创建了 model 并放在 GPU，下面直接复用 model
results = []
for i in range(5):
    ckpt = torch.load(save_model_path + f"pertkg{i}.pt", map_location="cpu")
    model.load_state_dict(ckpt)
    model.normalize_parameters()
    model.eval()

    with torch.no_grad():
        ent_emb, rel_emb = model.get_embeddings()  # 在 GPU
        score = inference(h_cand_ent,
                          ent2id, rel2id,
                          ent_emb, rel_emb,
                          h_cand, t_cand,
                          'batch_target_inference')

        score_np = to_cpu_numpy(score)
        results.append(score_np)

    del ckpt
    del ent_emb, rel_emb, score, score_np
    gc.collect()
    torch.cuda.empty_cache()

# 把 5 个结果做平均（results 内是 numpy 数组）
arrs = [np.array(x) for x in results]
average_arr = np.mean(arrs, axis=0).tolist()

ti_dict_n_n = {}
for idx, ent in enumerate(h_cand_ent):
    ti_dict_n_n[f'{ent}'] = average_arr[idx]

# 计算百分比（保持你原来的逻辑）
ti_percent_dict = {}
for k, v in tqdm.tqdm(ti_dict.items(), desc="compute confidence"):
    ti_percent = []
    k_ranks = get_rank(v)

    comp_ranks = []
    for comp in h_cand_ent:
        comp_ranks.append(get_rank(ti_dict_n_n[comp]))

    packed_ranks = list(zip(*comp_ranks))
    for idx, x in enumerate(k_ranks):
        ranks = packed_ranks[idx]
        ti_percent.append(sum([1 for i in ranks if i >= (x + 50)]) / len(ranks))

    ti_percent_dict[k] = ti_percent

# 输出文件保持不变
for k, v in ti_dict.items():
    ti_score = v
    ti_percent = ti_percent_dict[k]
    df = pd.DataFrame({'target': t_cand_ent,
                       'ti_score': ti_score,
                       'confidence': ti_percent})
    df.to_csv(output_path + '{}.txt'.format(k), sep='\t', index=False, header=True)


device: cuda


compute confidence: 100%|██████████| 20/20 [13:45<00:00, 41.30s/it]


In [6]:
# output 
for k,v in ti_dict.items():
    ti_score = v
    ti_percent = ti_percent_dict[k]
    df = pd.DataFrame({'target':t_cand_ent,
                       'ti_score':ti_score,
                       'confidence':ti_percent})
    df.to_csv(output_path + '{}.txt'.format(k),sep='\t',index=False,header=True)